# Political Response Clarity with DeBERTa-v3

Fine-tuning microsoft/deberta-v3-base on QEvasion question-answer pairs using dynamic padding, class-aware loss, mixed precision, checkpointing, and structured error analysis.

## Results status

The source was preserved, but the Kaggle export did not retain verifiable outputs. Complete a clean rerun before adding any metrics, plots, or model-comparison claims.

## Design Rationale

Input construction: the main run uses the question and answer as a transformer sentence pair, equivalent to `[CLS] question [SEP] answer [SEP]`. This keeps the two fields structurally separated and lets cross-attention compare the answer against the actual question.

Tokenizer/model loading: `AutoTokenizer` and `AutoModelForSequenceClassification` are used so the same code path can be reused for BERT, DistilBERT, and DeBERTa. For DeBERTa-v3, the tokenizer is SentencePiece based, so `sentencepiece` must be available and `protobuf` must stay in a Kaggle-compatible range (`>=5.29.1,<6`).

Data handling: CSV files are loaded through Hugging Face `load_dataset`, converted into split-specific `Dataset` objects, and tokenized with batched `map()` before entering PyTorch. Dynamic padding is applied by `DataCollatorWithPadding`, avoiding wasteful padding to max length for every example.

In [ ]:
import os
# Keep TensorFlow/JAX/XLA side libraries quiet and prevent Transformers from importing them.
# These must be set before importing transformers/evaluate/torch-adjacent packages.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import glob
import json
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*cuda capability.*")
warnings.filterwarnings("ignore", message=".*Tesla P100.*")
warnings.filterwarnings("ignore", message=".*not compatible with the current PyTorch installation.*")
warnings.filterwarnings("ignore", category=UserWarning, module="torch.cuda")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import evaluate

from datasets import Dataset, DatasetDict, load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from transformers.utils import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 180)

In [ ]:
# ============================================================
# Configuration
# ============================================================

MODEL_CHECKPOINT = "microsoft/deberta-v3-base"
SUBMISSION_FILE = "submission_microsoft-deberta-v3-base.csv"

# Leave these as None for automatic schema detection, or set manually if needed.
TRAIN_PATH_OVERRIDE = None
TEST_PATH_OVERRIDE = None
SAMPLE_SUBMISSION_PATH_OVERRIDE = None
QUESTION_COL_OVERRIDE = None
ANSWER_COL_OVERRIDE = None
LABEL_COL_OVERRIDE = None
ID_COL_OVERRIDE = None

# Main input formulation. To run an input-formulation ablation, switch to "structured_text"
# and rerun the notebook; the experiment summary cell will record the setting.
INPUT_FORMAT = "sentence_pair"  # "sentence_pair" or "structured_text"

SEED = 42
VALIDATION_SIZE = 0.15
# Faster DeBERTa default. 256 usually preserves most Q/A signal while cutting attention cost sharply.
MAX_LENGTH = 256
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_ACCUM_STEPS = 1
GRAD_CLIP_NORM = 1.0
PATIENCE = 1
LABEL_SMOOTHING = 0.02
USE_CLASS_WEIGHTS = True
PRIMARY_METRIC = "weighted_f1"  # Kaggle reports F1; macro_f1 is also reported below.

# Speed controls. Keep FAST_DEV_RUN=False for a real Kaggle submission.
# Set FAST_DEV_RUN=True only for a quick notebook smoke test.
FAST_DEV_RUN = False
FAST_TRAIN_SIZE = 768
FAST_VALID_SIZE = 256
FAST_TEST_SIZE = 256

# During training, validate on a stratified subset to save time; later cells run full validation.
FAST_VALIDATION_DURING_TRAINING = True
TRAIN_VALIDATION_SIZE = 512

# If Kaggle shows "CUDA error: no kernel image is available", leave this as False;
# the smoke tests below will automatically fall back to CPU before training.
FORCE_CPU = False
ALLOW_CUDA = True

OUTPUT_DIR = Path(".")
BEST_MODEL_PATH = OUTPUT_DIR / "best_microsoft-deberta-v3-base.pt"
HISTORY_CSV = OUTPUT_DIR / "deberta_training_history.csv"


def cuda_basic_smoke_test():
    """Return True only if this PyTorch build can actually execute on the Kaggle GPU."""
    if FORCE_CPU or not ALLOW_CUDA:
        return False

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if not torch.cuda.is_available():
            return False
        try:
            device_name = torch.cuda.get_device_name(0)
            major, minor = torch.cuda.get_device_capability(0)
        except Exception as exc:
            print("CUDA device query failed; using CPU instead.")
            print("CUDA failure:", repr(exc))
            return False

    # Kaggle P100 is sm_60. Current Kaggle/PyTorch builds may support only sm_70+,
    # which produces: "no kernel image is available for execution on the device".
    if major < 7:
        print(
            f"Detected {device_name} with CUDA capability sm_{major}{minor}. "
            "This PyTorch build does not support P100/sm_60, so the notebook will use CPU. "
            "For fast DeBERTa training, switch the Kaggle accelerator to T4 x2 or L4."
        )
        return False

    try:
        x = torch.randn((8, 8), device="cuda")
        y = (x @ x.T).sum()
        _ = y.item()
        torch.cuda.synchronize()
        return True
    except Exception as exc:
        print("CUDA basic smoke test failed; using CPU instead.")
        print("CUDA failure:", repr(exc))
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False

DEVICE = torch.device("cuda" if cuda_basic_smoke_test() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
PAD_TO_MULTIPLE_OF = 8 if DEVICE.type == "cuda" else None

print("Model:", MODEL_CHECKPOINT)
print("Device:", DEVICE)
print("AMP enabled:", USE_AMP)
if torch.cuda.is_available():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            print("CUDA device:", torch.cuda.get_device_name(0))
            print("CUDA capability:", torch.cuda.get_device_capability(0))
        except Exception as exc:
            print("CUDA device name unavailable:", repr(exc))

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        # Full bitwise determinism slows transformer fine-tuning substantially.
        # The random seeds still make the split and initialization reproducible enough for this experiment.
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass


seed_everything(SEED)

## Data Loading

The notebook searches Kaggle input directories and the current working directory for CSV or Parquet data. If local files are not found, it falls back to the pinned public Hugging Face QEvasion Parquet files used for the CLARITY dataset.

In [ ]:
HF_DATASET_REPO = "ailsntua/QEvasion"
HF_DATASET_REVISION = "3afc18f0b582b3cfdb927822cff57ddc6e871f9c"
HF_DATA_BASE_URL = f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/{HF_DATASET_REVISION}/data"
HF_TRAIN_URL = f"{HF_DATA_BASE_URL}/train-00000-of-00001.parquet"
HF_TEST_URL = f"{HF_DATA_BASE_URL}/test-00000-of-00001.parquet"


def find_data_files():
    patterns = [
        "/kaggle/input/**/*.csv",
        "/kaggle/input/**/*.parquet",
        "./**/*.csv",
        "./**/*.parquet",
    ]
    files = []
    for pattern in patterns:
        files.extend(glob.glob(pattern, recursive=True))
    return sorted(set(files))


def choose_data_file(files, exact_names, contains_names, excluded_names=()):
    by_name = [(path, Path(path).name.lower()) for path in files]
    excluded_names = tuple(name.lower() for name in excluded_names)

    def allowed(name):
        return not any(excluded in name for excluded in excluded_names)

    for exact in exact_names:
        exact = exact.lower()
        for path, name in by_name:
            if name == exact and allowed(name):
                return path

    for needle in contains_names:
        needle = needle.lower()
        for path, name in by_name:
            if needle in name and allowed(name):
                return path

    return None


def load_table_pair(train_path, test_path):
    train_suffix = Path(train_path).suffix.lower()
    test_suffix = Path(test_path).suffix.lower()

    if train_suffix == ".csv" and test_suffix == ".csv":
        raw = load_dataset("csv", data_files={"train_full": train_path, "test": test_path})
    elif train_suffix == ".parquet" and test_suffix == ".parquet":
        raw = load_dataset("parquet", data_files={"train_full": train_path, "test": test_path})
    else:
        # Mixed local formats are rare; pandas handles this fallback cleanly.
        train_table = pd.read_csv(train_path) if train_suffix == ".csv" else pd.read_parquet(train_path)
        test_table = pd.read_csv(test_path) if test_suffix == ".csv" else pd.read_parquet(test_path)
        return train_table, test_table, "mixed local files"

    return raw["train_full"].to_pandas(), raw["test"].to_pandas(), f"local {train_suffix}/{test_suffix} via load_dataset"


def load_sample_submission(path):
    if path is None or not os.path.exists(path):
        return None
    if Path(path).suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_parquet(path)


data_files = find_data_files()
print("Data files found:")
for file in data_files[:80]:
    print(" ", file)
if len(data_files) > 80:
    print(f"  ... {len(data_files) - 80} more files")

TRAIN_PATH = TRAIN_PATH_OVERRIDE or choose_data_file(
    data_files,
    ["train.csv", "train.parquet", "train-00000-of-00001.parquet"],
    ["train"],
    excluded_names=["sample", "submission"],
)
TEST_PATH = TEST_PATH_OVERRIDE or choose_data_file(
    data_files,
    ["test.csv", "test.parquet", "test-00000-of-00001.parquet"],
    ["test"],
    excluded_names=["sample", "submission"],
)
SAMPLE_SUBMISSION_PATH = SAMPLE_SUBMISSION_PATH_OVERRIDE or choose_data_file(
    data_files,
    ["sample_submission.csv", "sample_submission.parquet"],
    ["sample_submission", "submission", "sample"],
)

if TRAIN_PATH is not None and TEST_PATH is not None:
    train_df, test_df, data_source = load_table_pair(TRAIN_PATH, TEST_PATH)
else:
    print("No local train/test pair found; trying pinned Hugging Face Parquet files.")
    print("HF_TRAIN_URL:", HF_TRAIN_URL)
    print("HF_TEST_URL:", HF_TEST_URL)
    try:
        raw = load_dataset(
            "parquet",
            data_files={"train_full": HF_TRAIN_URL, "test": HF_TEST_URL},
        )
        train_df = raw["train_full"].to_pandas()
        test_df = raw["test"].to_pandas()
        TRAIN_PATH = HF_TRAIN_URL
        TEST_PATH = HF_TEST_URL
        data_source = f"Hugging Face {HF_DATASET_REPO}@{HF_DATASET_REVISION[:7]}"
    except Exception as exc:
        raise FileNotFoundError(
            "Could not infer train/test data files locally, and the Hugging Face fallback failed. "
            "Attach the Kaggle dataset to this notebook or set TRAIN_PATH_OVERRIDE and TEST_PATH_OVERRIDE "
            "to the exact CSV/Parquet paths printed above."
        ) from exc

sample_submission = load_sample_submission(SAMPLE_SUBMISSION_PATH)

print("Data source:", data_source)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH)
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("train columns:", train_df.columns.tolist())
print("test columns:", test_df.columns.tolist())
if sample_submission is not None:
    print("sample submission columns:", sample_submission.columns.tolist())
else:
    print("sample submission: not found; will create Id/Predicted file")

display(train_df.head(3))

In [ ]:
def pick_column(columns, candidates):
    lower_to_original = {c.lower(): c for c in columns}
    for candidate in candidates:
        if candidate is None:
            continue
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]
    return None


QUESTION_COL = QUESTION_COL_OVERRIDE or pick_column(
    train_df.columns,
    ["question", "Question", "questions", "prompt", "query"],
)
ANSWER_COL = ANSWER_COL_OVERRIDE or pick_column(
    train_df.columns,
    ["interview_answer", "answer", "Answer", "response", "reply", "text"],
)
LABEL_COL = LABEL_COL_OVERRIDE or pick_column(
    train_df.columns,
    ["clarity_label", "label", "Label", "target", "Predicted", "class"],
)

if sample_submission is not None:
    sample_id_candidate = sample_submission.columns[0]
else:
    sample_id_candidate = None

ID_COL = ID_COL_OVERRIDE or pick_column(
    test_df.columns,
    [sample_id_candidate, "Id", "id", "ID", "index", "Index"],
)

missing = [
    name
    for name, value in {
        "QUESTION_COL": QUESTION_COL,
        "ANSWER_COL": ANSWER_COL,
        "LABEL_COL": LABEL_COL,
    }.items()
    if value is None
]
if missing:
    raise ValueError(f"Could not infer required columns: {missing}. Set the *_OVERRIDE variables.")

EXPECTED_LABEL_ORDER = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
observed_labels = train_df[LABEL_COL].astype(str).unique().tolist()
if set(EXPECTED_LABEL_ORDER).issubset(set(observed_labels)):
    label_names = EXPECTED_LABEL_ORDER
else:
    label_names = sorted(observed_labels)

label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

for col in [QUESTION_COL, ANSWER_COL]:
    train_df[col] = train_df[col].fillna("").astype(str)
    test_df[col] = test_df[col].fillna("").astype(str)

train_df[LABEL_COL] = train_df[LABEL_COL].astype(str)
train_df["labels"] = train_df[LABEL_COL].map(label2id)

if train_df["labels"].isna().any():
    unknown = sorted(train_df.loc[train_df["labels"].isna(), LABEL_COL].unique().tolist())
    raise ValueError(f"Unknown labels not present in label2id: {unknown}")

train_df["labels"] = train_df["labels"].astype("int64")
train_df["row_id"] = np.arange(len(train_df))
test_df["row_id"] = np.arange(len(test_df))

print("QUESTION_COL:", QUESTION_COL)
print("ANSWER_COL:", ANSWER_COL)
print("LABEL_COL:", LABEL_COL)
print("ID_COL:", ID_COL)
print("label2id:", label2id)
display(train_df[LABEL_COL].value_counts().rename("count").to_frame())

## Exploratory Analysis

These plots support the report section on data distribution and potential sources of class imbalance or length-related difficulty.

In [ ]:
train_df["question_word_len"] = train_df[QUESTION_COL].str.split().str.len()
train_df["answer_word_len"] = train_df[ANSWER_COL].str.split().str.len()
train_df["combined_word_len"] = train_df["question_word_len"] + train_df["answer_word_len"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = train_df[LABEL_COL].value_counts().reindex(label_names)
axes[0].bar(class_counts.index, class_counts.values, color=["#4C78A8", "#F58518", "#54A24B"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Examples")
axes[0].tick_params(axis="x", rotation=20)
for i, count in enumerate(class_counts.values):
    axes[0].text(i, count, str(int(count)), ha="center", va="bottom")

for label in label_names:
    subset = train_df[train_df[LABEL_COL] == label]
    axes[1].hist(subset["answer_word_len"], bins=30, alpha=0.45, label=label)
axes[1].set_title("Answer Length by Class")
axes[1].set_xlabel("Answer length in words")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.savefig("deberta_data_overview.png", dpi=160, bbox_inches="tight")
plt.show()

display(
    train_df.groupby(LABEL_COL)[["question_word_len", "answer_word_len", "combined_word_len"]]
    .agg(["mean", "median", "max"])
    .round(2)
)

## Validation Split and Hugging Face Dataset Construction

A stratified validation split is used so each clarity class is represented in validation. Tokenization is done with batched `Dataset.map()` before PyTorch training.

In [ ]:
train_indices, valid_indices = train_test_split(
    np.arange(len(train_df)),
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=train_df["labels"].values,
)

train_split_df = train_df.iloc[train_indices].reset_index(drop=True)
valid_split_df = train_df.iloc[valid_indices].reset_index(drop=True)

if FAST_DEV_RUN:
    train_split_df = train_split_df.groupby("labels", group_keys=False).sample(
        n=max(1, min(FAST_TRAIN_SIZE // len(label_names), train_split_df["labels"].value_counts().min())),
        random_state=SEED,
    ).reset_index(drop=True)
    valid_split_df = valid_split_df.groupby("labels", group_keys=False).sample(
        n=max(1, min(FAST_VALID_SIZE // len(label_names), valid_split_df["labels"].value_counts().min())),
        random_state=SEED,
    ).reset_index(drop=True)
    test_df = test_df.head(FAST_TEST_SIZE).reset_index(drop=True)
    print("FAST_DEV_RUN enabled: using small stratified train/validation subsets.")

train_eval_df = valid_split_df
if FAST_VALIDATION_DURING_TRAINING and len(valid_split_df) > TRAIN_VALIDATION_SIZE:
    per_class_n = max(1, TRAIN_VALIDATION_SIZE // len(label_names))
    train_eval_df = valid_split_df.groupby("labels", group_keys=False).sample(
        n=min(per_class_n, valid_split_df["labels"].value_counts().min()),
        random_state=SEED,
    ).reset_index(drop=True)
    print(f"Training-time validation subset: {len(train_eval_df)} / {len(valid_split_df)} examples")

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_split_df, preserve_index=False),
    "validation": Dataset.from_pandas(valid_split_df, preserve_index=False),
    "train_validation": Dataset.from_pandas(train_eval_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

print("train split:", len(raw_datasets["train"]))
print("training-time validation split:", len(raw_datasets["train_validation"]))
print("full validation split:", len(raw_datasets["validation"]))
print("test split:", len(raw_datasets["test"]))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    use_fast=True,
)

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length reported by tokenizer:", tokenizer.model_max_length)
print("Special tokens:", tokenizer.special_tokens_map)


def make_text_inputs(batch):
    questions = ["" if value is None else str(value) for value in batch[QUESTION_COL]]
    answers = ["" if value is None else str(value) for value in batch[ANSWER_COL]]

    if INPUT_FORMAT == "sentence_pair":
        return questions, answers

    if INPUT_FORMAT == "structured_text":
        combined = [
            f"Question: {q}\nAnswer: {a}"
            for q, a in zip(questions, answers)
        ]
        return combined, None

    raise ValueError(f"Unknown INPUT_FORMAT: {INPUT_FORMAT}")


def tokenize_batch(batch):
    text_a, text_b = make_text_inputs(batch)
    if text_b is None:
        return tokenizer(text_a, truncation=True, max_length=MAX_LENGTH)
    return tokenizer(text_a, text_b, truncation=True, max_length=MAX_LENGTH)


columns_to_remove = {}
for split in raw_datasets:
    keep = {"labels"} if split != "test" else set()
    columns_to_remove[split] = [
        col for col in raw_datasets[split].column_names
        if col not in keep
    ]

tokenized_datasets = DatasetDict()
for split in ["train", "train_validation", "validation", "test"]:
    tokenized_datasets[split] = raw_datasets[split].map(
        tokenize_batch,
        batched=True,
        remove_columns=columns_to_remove[split],
        desc=f"Tokenizing {split}",
    )

NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=PAD_TO_MULTIPLE_OF,
)

train_loader = DataLoader(
    tokenized_datasets["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

train_valid_loader = DataLoader(
    tokenized_datasets["train_validation"],
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

valid_loader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

test_loader = DataLoader(
    tokenized_datasets["test"],
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
)

### About the DeBERTa Load Report

When `microsoft/deberta-v3-base` is loaded as `AutoModelForSequenceClassification`, the masked-language-modeling head from pretraining is discarded and a new pooler/classifier head is initialized for the three CLARITY labels. Messages about `lm_predictions` being unexpected and `classifier`/`pooler` being missing are expected for fine-tuning and do not indicate a broken model. The notebook suppresses the verbose Transformers load report and trains the new classification head below.

## Model, Optimizer, and Loss

The training loop is fully manual: PyTorch `DataLoader`, AdamW, linear warmup/decay, gradient clipping, optional class weights, and AMP mixed precision on CUDA. The Hugging Face `Trainer` API is not used.

In [ ]:
def load_sequence_classifier(device):
    loaded_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(label_names),
        id2label=id2label,
        label2id=label2id,
        use_safetensors=True,
    )
    loaded_model.to(device)
    return loaded_model


def deberta_forward_smoke_test(loaded_model):
    """Check that the selected device can run a real DeBERTa forward pass."""
    if DEVICE.type != "cuda":
        return True
    try:
        loaded_model.eval()
        sample_batch = next(iter(train_loader))
        sample_inputs = {
            key: value[:1].to(DEVICE)
            for key, value in sample_batch.items()
            if key != "labels"
        }
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                _ = loaded_model(**sample_inputs).logits
        torch.cuda.synchronize()
        return True
    except Exception as exc:
        print("CUDA DeBERTa forward smoke test failed; switching to CPU.")
        print("CUDA failure:", repr(exc))
        return False


model = load_sequence_classifier(DEVICE)

if DEVICE.type == "cuda" and not deberta_forward_smoke_test(model):
    # Some Kaggle GPU/PyTorch combinations fail with
    # "no kernel image is available for execution on the device". CPU is slower,
    # but it lets the notebook complete and produce a valid submission.
    del model
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    DEVICE = torch.device("cpu")
    USE_AMP = False
    PIN_MEMORY = False
    PAD_TO_MULTIPLE_OF = None
    model = load_sequence_classifier(DEVICE)
    print("Runtime device after fallback:", DEVICE)

label_counts = train_split_df["labels"].value_counts().sort_index()
if USE_CLASS_WEIGHTS:
    weights = len(train_split_df) / (len(label_names) * label_counts)
    class_weights = torch.tensor(
        [weights.get(i, 1.0) for i in range(len(label_names))],
        dtype=torch.float,
        device=DEVICE,
    )
else:
    class_weights = None

print("class weights:", None if class_weights is None else class_weights.detach().cpu().numpy())

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING,
)

no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
optimizer_grouped_parameters = [
    {
        "params": [
            p for n, p in model.named_parameters()
            if not any(nd in n for nd in no_decay)
        ],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [
            p for n, p in model.named_parameters()
            if any(nd in n for nd in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
num_training_steps = EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("training steps:", num_training_steps)
print("warmup steps:", num_warmup_steps)
print("final training device:", DEVICE)
print("final AMP enabled:", USE_AMP)

In [ ]:
def move_to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def split_inputs_and_labels(batch):
    labels = batch.get("labels")
    inputs = {key: value for key, value in batch.items() if key != "labels"}
    return inputs, labels


def compute_loss_and_logits(batch):
    inputs, labels = split_inputs_and_labels(batch)
    outputs = model(**inputs)
    if labels is None:
        return None, outputs.logits
    loss = loss_fn(outputs.logits.float(), labels)
    return loss, outputs.logits


def compute_metrics(y_true, y_pred):
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
    }


def predict_with_loss(loader, desc="predict"):
    model.eval()
    losses = []
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, leave=False):
            batch = move_to_device(batch)
            labels = batch.get("labels")

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                loss, logits = compute_loss_and_logits(batch)

            if loss is not None:
                losses.append(loss.item())
            all_logits.append(logits.detach().cpu())
            if labels is not None:
                all_labels.append(labels.detach().cpu())

    logits = torch.cat(all_logits).numpy()
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)

    labels = None
    if all_labels:
        labels = torch.cat(all_labels).numpy()

    mean_loss = float(np.mean(losses)) if losses else None
    return preds, probs, labels, mean_loss

## Manual Training Loop

The best checkpoint is selected by validation weighted F1. Macro F1, precision, recall, and accuracy are also logged for analysis.

In [ ]:
import time

history = []
best_score = -1.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(progress, start=1):
        batch = move_to_device(batch)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            loss, _ = compute_loss_and_logits(batch)
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_for_backward).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item()
        progress.set_postfix(train_loss=running_loss / step)

    val_preds, val_probs, val_labels, val_loss = predict_with_loss(train_valid_loader, desc="validation subset")
    metrics = compute_metrics(val_labels, val_preds)

    row = {
        "epoch": epoch,
        "train_loss": running_loss / max(1, len(train_loader)),
        "validation_loss": val_loss,
        "epoch_minutes": (time.time() - epoch_start) / 60,
        **metrics,
    }
    history.append(row)
    history_df = pd.DataFrame(history)
    history_df.to_csv(HISTORY_CSV, index=False)

    print(
        f"epoch={epoch} "
        f"train_loss={row['train_loss']:.4f} "
        f"val_loss={row['validation_loss']:.4f} "
        f"weighted_f1={row['weighted_f1']:.4f} "
        f"macro_f1={row['macro_f1']:.4f} "
        f"acc={row['accuracy']:.4f} "
        f"minutes={row['epoch_minutes']:.1f}"
    )

    current_score = row[PRIMARY_METRIC]
    if current_score > best_score:
        best_score = current_score
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label2id": label2id,
                "id2label": id2label,
                "config": {
                    "checkpoint": MODEL_CHECKPOINT,
                    "input_format": INPUT_FORMAT,
                    "max_length": MAX_LENGTH,
                    "seed": SEED,
                },
                "best_epoch": epoch,
                "best_score": best_score,
            },
            BEST_MODEL_PATH,
        )
        print("saved best checkpoint:", BEST_MODEL_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break

history_df = pd.DataFrame(history)
display(history_df)

## Training Curves

Use these plots to discuss underfitting/overfitting. A widening gap between improving train loss and declining validation F1 is evidence of overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train loss")
axes[0].plot(history_df["epoch"], history_df["validation_loss"], marker="o", label="validation loss")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["weighted_f1"], marker="o", label="weighted F1")
axes[1].plot(history_df["epoch"], history_df["macro_f1"], marker="o", label="macro F1")
axes[1].set_title("Validation F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("deberta_training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## Final Validation Evaluation

This section reports the metrics required for analysis: accuracy, precision, recall, weighted F1, and macro F1.

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

val_preds, val_probs, val_labels, val_loss = predict_with_loss(valid_loader, desc="best validation")
final_metrics = compute_metrics(val_labels, val_preds)

print("Best epoch:", checkpoint["best_epoch"])
print("Validation loss:", round(val_loss, 4))
for key, value in final_metrics.items():
    print(f"{key}: {value:.4f}")

report_dict = classification_report(
    val_labels,
    val_preds,
    labels=list(range(len(label_names))),
    target_names=label_names,
    digits=4,
    zero_division=0,
    output_dict=True,
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv("deberta_classification_report.csv")
display(report_df)

try:
    hf_f1 = evaluate.load("f1")
    print(
        "HF evaluate weighted F1:",
        hf_f1.compute(predictions=val_preds, references=val_labels, average="weighted")["f1"],
    )
    print(
        "HF evaluate macro F1:",
        hf_f1.compute(predictions=val_preds, references=val_labels, average="macro")["f1"],
    )
except Exception as exc:
    print("HF evaluate cross-check skipped:", repr(exc))

In [ ]:
cm = confusion_matrix(val_labels, val_preds, labels=list(range(len(label_names))))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

fig, ax = plt.subplots(figsize=(7.5, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("DeBERTa-v3 Validation Confusion Matrix")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("deberta_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

per_class = report_df.loc[label_names, ["precision", "recall", "f1-score"]]
per_class.plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=20)
plt.title("Per-Class Validation Metrics")
plt.ylabel("Score")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("deberta_per_class_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

## Error Analysis

The next cells identify the hardest class, recurring confusion patterns, and concrete validation failures to include in the report.

In [ ]:
valid_analysis_df = valid_split_df.copy()
valid_analysis_df["true_label"] = [id2label[int(i)] for i in val_labels]
valid_analysis_df["pred_label"] = [id2label[int(i)] for i in val_preds]
valid_analysis_df["correct"] = valid_analysis_df["true_label"] == valid_analysis_df["pred_label"]
valid_analysis_df["confidence"] = val_probs.max(axis=1)
valid_analysis_df["question_word_len"] = valid_analysis_df[QUESTION_COL].str.split().str.len()
valid_analysis_df["answer_word_len"] = valid_analysis_df[ANSWER_COL].str.split().str.len()

errors_df = valid_analysis_df.loc[~valid_analysis_df["correct"]].copy()

print(f"Validation errors: {len(errors_df)} / {len(valid_analysis_df)}")

hardest = (
    valid_analysis_df.groupby("true_label")["correct"]
    .agg(error_rate=lambda s: 1.0 - s.mean(), count="size")
    .sort_values("error_rate", ascending=False)
)
display(hardest)

confusion_patterns = (
    errors_df.groupby(["true_label", "pred_label"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values("count", ascending=False)
)
display(confusion_patterns.head(10))

example_cols = [
    "true_label",
    "pred_label",
    "confidence",
    QUESTION_COL,
    ANSWER_COL,
]
display(
    errors_df.sort_values("confidence", ascending=False)[example_cols]
    .head(12)
)

## Subgroup Analysis by Question and Answer Length

This supports the analysis goal to inspect performance across data subgroups. Low scores in long-answer buckets often indicate difficulty tracking whether the answer actually addresses the question.

In [ ]:
valid_analysis_df["question_len_bucket"] = pd.cut(
    valid_analysis_df["question_word_len"],
    bins=[-1, 10, 20, 40, 10_000],
    labels=["q <= 10", "10 < q <= 20", "20 < q <= 40", "q > 40"],
)
valid_analysis_df["answer_len_bucket"] = pd.cut(
    valid_analysis_df["answer_word_len"],
    bins=[-1, 50, 100, 200, 10_000],
    labels=["a <= 50", "50 < a <= 100", "100 < a <= 200", "a > 200"],
)


def subgroup_metrics(df, group_col):
    rows = []
    for group_name, group in df.groupby(group_col, observed=False):
        if len(group) == 0:
            continue
        y_true = group["true_label"].map(label2id).values
        y_pred = group["pred_label"].map(label2id).values
        metrics = compute_metrics(y_true, y_pred)
        rows.append({
            "group": str(group_name),
            "n": len(group),
            "accuracy": metrics["accuracy"],
            "weighted_f1": metrics["weighted_f1"],
            "macro_f1": metrics["macro_f1"],
        })
    return pd.DataFrame(rows)


question_subgroups = subgroup_metrics(valid_analysis_df, "question_len_bucket")
answer_subgroups = subgroup_metrics(valid_analysis_df, "answer_len_bucket")
question_subgroups.to_csv("deberta_question_length_subgroups.csv", index=False)
answer_subgroups.to_csv("deberta_answer_length_subgroups.csv", index=False)

display(question_subgroups)
display(answer_subgroups)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, table, title in [
    (axes[0], question_subgroups, "Question Length Subgroups"),
    (axes[1], answer_subgroups, "Answer Length Subgroups"),
]:
    x = np.arange(len(table))
    width = 0.35
    ax.bar(x - width / 2, table["accuracy"], width=width, label="accuracy")
    ax.bar(x + width / 2, table["macro_f1"], width=width, label="macro F1")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(table["group"], rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)
    for idx, row in table.iterrows():
        ax.text(idx, 0.03, f"n={int(row['n'])}", ha="center", fontsize=9)
    ax.legend()

plt.tight_layout()
plt.savefig("deberta_subgroup_analysis.png", dpi=160, bbox_inches="tight")
plt.show()

## Experiment Summary Row

After each run, keep this CSV and copy its numbers into the final cross-model comparison table. Rerun with different `MAX_LENGTH`, `LEARNING_RATE`, `INPUT_FORMAT`, or batch settings to study hyperparameter and input-formulation effects.

In [ ]:
experiment_summary = pd.DataFrame([
    {
        "model": MODEL_CHECKPOINT,
        "input_format": INPUT_FORMAT,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
        "fast_validation_during_training": FAST_VALIDATION_DURING_TRAINING,
        "train_validation_size": len(train_eval_df),
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "class_weights": USE_CLASS_WEIGHTS,
        "best_epoch": checkpoint["best_epoch"],
        "validation_loss": val_loss,
        **final_metrics,
        "submission_file": SUBMISSION_FILE,
    }
])

experiment_summary.to_csv("deberta_experiment_summary.csv", index=False)
display(experiment_summary)

## Test Prediction and Export

The submission file follows the required two-column format: `Id`, `Predicted`.

In [ ]:
test_preds, test_probs, _, _ = predict_with_loss(test_loader, desc="test")
predicted_labels = [id2label[int(idx)] for idx in test_preds]


def build_submission(test_frame, predicted_labels, sample_submission=None):
    if len(test_frame) != len(predicted_labels):
        raise ValueError(
            f"Prediction count {len(predicted_labels)} does not match test rows {len(test_frame)}"
        )

    if sample_submission is not None:
        if len(sample_submission) != len(predicted_labels):
            raise ValueError("sample_submission length does not match predictions.")
        submission = sample_submission.copy()
        if "Id" not in submission.columns:
            submission = submission.rename(columns={submission.columns[0]: "Id"})
        submission["Predicted"] = predicted_labels
        return submission[["Id", "Predicted"]]

    if ID_COL is not None:
        ids = test_frame[ID_COL].values
    else:
        ids = np.arange(len(test_frame))
    return pd.DataFrame({"Id": ids, "Predicted": predicted_labels})


submission = build_submission(test_df, predicted_labels, sample_submission)
submission.to_csv(SUBMISSION_FILE, index=False)

print("Saved:", SUBMISSION_FILE)
print("Rows:", len(submission))
print(submission["Predicted"].value_counts())
display(submission.head(10))

## Runtime notes

DeBERTa-v3 requires SentencePiece and a compatible protobuf release. GPU memory can be reduced with a shorter maximum sequence length, a smaller batch, or gradient accumulation. Public model and dataset downloads do not require credentials.